In [1]:
#applying Chronis and Erk's clustering method for the semantic similarity task on the SimLex-999 dataset

In [2]:
import os
import csv
import torch
import datasets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import spearmanr
from scipy.spatial.distance import cosine

In [3]:
from pathlib import Path

token = next(l.strip() for l in Path("hf_token.txt").read_text().splitlines() if l.strip().startswith("hf_"))
os.environ["HF_TOKEN"] = token
os.environ["HF_HOME"] = "D:/huggingface_cache"

In [4]:
print(torch.cuda.is_available())

True


In [5]:
from transformers import AutoModel, AutoTokenizer

MODEL_DIR = Path("D:/252 Project/models/gpt2-large-hf")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModel.from_pretrained(
    MODEL_DIR,
    dtype=torch.bfloat16,
    device_map="auto",
)

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

In [6]:
#checking number of layers for analysis later:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(MODEL_DIR)
print(f"Layers: {config.num_hidden_layers}")
print(f"Hidden Size: {config.hidden_size}")

Layers: 36
Hidden Size: 1280


In [9]:
#Get SimLex999 words for comparison
simlex999 = datasets.get_simlex999()
display(simlex999)

processed 999 word pairs from simlex999 dataset


[{'word1': 'old',
  'word2': 'new',
  'POS': 'A',
  'SimLex999': '1.58',
  'conc_w1': 2.72,
  'conc_w2': 2.81,
  'concQ': '2',
  'assoc_USF': '7.25',
  'sim_assoc333': '1',
  'SD_simlex': '0.41',
  'similarity': 1.58},
 {'word1': 'smart',
  'word2': 'intelligent',
  'POS': 'A',
  'SimLex999': '9.2',
  'conc_w1': 1.75,
  'conc_w2': 2.46,
  'concQ': '1',
  'assoc_USF': '7.11',
  'sim_assoc333': '1',
  'SD_simlex': '0.67',
  'similarity': 9.2},
 {'word1': 'hard',
  'word2': 'difficult',
  'POS': 'A',
  'SimLex999': '8.77',
  'conc_w1': 3.76,
  'conc_w2': 2.21,
  'concQ': '2',
  'assoc_USF': '5.94',
  'sim_assoc333': '1',
  'SD_simlex': '1.19',
  'similarity': 8.77},
 {'word1': 'happy',
  'word2': 'cheerful',
  'POS': 'A',
  'SimLex999': '9.55',
  'conc_w1': 2.56,
  'conc_w2': 2.34,
  'concQ': '1',
  'assoc_USF': '5.85',
  'sim_assoc333': '1',
  'SD_simlex': '2.18',
  'similarity': 9.55},
 {'word1': 'hard',
  'word2': 'easy',
  'POS': 'A',
  'SimLex999': '0.95',
  'conc_w1': 3.76,
  'conc_

In [10]:
# get a list of all the words in simlex999
first_word = [row['word1'] for row in simlex999]
second_word = [row['word2'] for row in simlex999]
simlex999_wordlist = first_word + second_word

In [11]:
#Get vector representation of final layer for our wordlist
outputs = {}
for word in simlex999_wordlist:
    inputs = tokenizer(word, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs[word] = model(**inputs, output_hidden_states=True).last_hidden_state[0, -1, :]

In [14]:
# need cosine similarity between 'word1' and 'word2' in each dict in the simlex999 list
def get_model_similarities(outputs):
    model_similarities = []
    for row in simlex999:
        model_similarities.append(torch.nn.functional.cosine_similarity(outputs[row['word1']], outputs[row['word2']], dim=-1))
    model_similarities = [s.item() for s in model_similarities]
    return model_similarities
model_similarities = get_model_similarities(outputs)

In [16]:
#list of human similarity judgments for spearman's p correlation
human_similarities = [row['SimLex999'] for row in simlex999]
len(human_similarities)
human_similarities = [float(s) for s in human_similarities]
human_similarities

[1.58,
 9.2,
 8.77,
 9.55,
 0.95,
 8.75,
 9.17,
 1.23,
 9.58,
 8.93,
 1.03,
 8.42,
 0.58,
 7.78,
 1.38,
 0.55,
 9.57,
 0.95,
 9.47,
 8.05,
 6.83,
 0.6,
 9.7,
 6.67,
 8.63,
 9.02,
 1.28,
 1.18,
 9.4,
 0.87,
 8.47,
 8.72,
 5.0,
 0.72,
 9.2,
 7.62,
 0.95,
 8.05,
 6.38,
 6.5,
 8.27,
 7.27,
 9.55,
 0.67,
 6.03,
 6.73,
 3.17,
 0.63,
 1.6,
 0.75,
 0.35,
 0.87,
 7.37,
 0.65,
 1.45,
 1.18,
 0.52,
 4.1,
 8.42,
 8.97,
 1.08,
 7.25,
 8.82,
 8.18,
 5.5,
 9.17,
 5.9,
 2.38,
 3.57,
 6.18,
 2.47,
 9.37,
 4.28,
 4.2,
 0.73,
 0.23,
 0.55,
 2.0,
 1.12,
 1.17,
 0.6,
 7.63,
 2.65,
 8.05,
 1.17,
 8.65,
 8.48,
 5.07,
 8.02,
 8.25,
 6.58,
 5.95,
 5.4,
 3.57,
 6.98,
 5.9,
 1.6,
 7.78,
 5.9,
 7.05,
 3.97,
 1.97,
 2.07,
 0.48,
 0.58,
 0.4,
 0.98,
 0.4,
 0.48,
 0.48,
 0.3,
 2.3,
 6.35,
 3.17,
 1.88,
 2.2,
 3.65,
 5.5,
 8.33,
 0.7,
 8.78,
 9.35,
 6.67,
 2.88,
 8.1,
 3.33,
 7.07,
 7.12,
 9.25,
 8.87,
 7.85,
 3.68,
 7.88,
 1.75,
 9.47,
 6.43,
 7.53,
 3.27,
 2.47,
 2.98,
 9.52,
 5.63,
 2.38,
 9.2,
 5.53,
 3.4,
 7.58,

In [18]:
#pass corresponding list items to spearmanr
spearmanr_correlations = spearmanr(model_similarities, human_similarities)
print(spearmanr_correlations)

SignificanceResult(statistic=np.float64(0.15164907544327025), pvalue=np.float64(1.4717093941663139e-06))


In [19]:
#next we try contextual embeddings with a simple template sentence: the "The {word} is here."

In [20]:
outputs = {}
for word in simlex999_wordlist:
    sentence = f"The {word} is here."
    word_start = sentence.index(word)
    word_end = word_start + len(word)

    enc = tokenizer(sentence, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc["offset_mapping"][0]

    token_indices = [
    i for i, (s, e) in enumerate(offsets.tolist())
    if s < word_end and e > word_start and s < e
    ]

    inputs = {k: v.to(model.device) for k, v in enc.items() if k != "offset_mapping"}
    with torch.no_grad():
        hidden = model(**inputs, output_hidden_states=True).last_hidden_state
        outputs[word] = hidden[0, token_indices, :].mean(dim=0)


In [22]:
model_similarities = get_model_similarities(outputs)

[0.60546875,
 0.6875,
 0.63671875,
 0.65625,
 0.5625,
 0.515625,
 0.515625,
 0.65625,
 0.76953125,
 0.8515625,
 0.6875,
 0.515625,
 0.6328125,
 0.46875,
 0.427734375,
 0.6328125,
 0.82421875,
 0.373046875,
 0.86328125,
 0.609375,
 0.578125,
 0.53125,
 0.55859375,
 0.625,
 0.82421875,
 0.82421875,
 0.546875,
 0.66015625,
 0.6640625,
 0.408203125,
 0.56640625,
 0.7890625,
 0.50390625,
 0.48828125,
 0.82421875,
 0.400390625,
 0.64453125,
 0.67578125,
 0.51953125,
 0.828125,
 0.419921875,
 0.318359375,
 0.62890625,
 0.490234375,
 0.7578125,
 0.470703125,
 0.388671875,
 0.640625,
 0.5234375,
 0.53515625,
 0.314453125,
 0.5703125,
 0.61328125,
 0.384765625,
 0.3984375,
 0.40625,
 0.5390625,
 0.447265625,
 0.64453125,
 0.6796875,
 0.578125,
 0.78515625,
 0.8828125,
 0.71875,
 0.640625,
 0.6953125,
 0.486328125,
 0.359375,
 0.421875,
 0.44140625,
 0.388671875,
 0.85546875,
 0.56640625,
 0.369140625,
 0.5703125,
 0.455078125,
 0.408203125,
 0.380859375,
 0.419921875,
 0.54296875,
 0.61328125,
 

In [23]:
spearmanr_correlations = spearmanr(model_similarities, human_similarities)
print(spearmanr_correlations)

SignificanceResult(statistic=np.float64(0.39289690516872555), pvalue=np.float64(3.2047431056426805e-38))


In [ ]:
#get contextual embeddings by getting the mean of embeddings from token activations using sentences from
#the BNC dataset

In [24]:
import datasets

def load_bnc():
    from nltk.corpus.reader import bnc
    bnc_reader = bnc.BNCCorpusReader(root='./data/BNC/Texts/', fileids=r'[A-K]/\w*/\w*\.xml')
    return bnc_reader #.tagged_sents()

In [25]:
def randomly(seq):
    import random
    shuffled = list(seq)
    random.shuffle(shuffled)
    return list(shuffled)

In [49]:
#build dict of word -> [list of sentences]
import importlib
importlib.reload(datasets)

bnc_reader = datasets.get_bnc()

def collect_bnc_examples(wordlist, max_num_examples=100):
    bnc_reader = datasets.get_bnc()
    print("BNC reader initialized, loading corpus...")
    corpus = bnc_reader.tagged_sents(strip_space=True)
    print("Corpus loaded, counting sentences...")
    corpus_length = len(corpus)
    print("# Sentences in BNC corpus: %s" % corpus_length)

    unigrams = {word: max_num_examples for word in wordlist}
    bnc_examples = {word: [] for word in wordlist}

    randomized_indexes = randomly([x for x in range(corpus_length - 1)])
    while unigrams and randomized_indexes:
        corpus_index = randomized_indexes.pop()
          
        if len(randomized_indexes) % 10000 == 0:
            print(f"Remaining: {len(randomized_indexes)} indexes, {len(unigrams)} words left")
              
        sentence = corpus[corpus_index]
          
        for word_tuple in sentence:
            word = word_tuple[0]
            if unigrams.get(word) is not None and unigrams[word] > 0:
                string = ' '.join([w[0] for w in sentence])
                bnc_examples[word].append(string)
                unigrams[word] -= 1
                if unigrams[word] == 0:
                    del unigrams[word]

    return bnc_examples

In [58]:
#getting average of all token vectors from the BNC-corpus sampled sentences for each word, across all layers
import warnings
import pickle, os

warnings.filterwarnings("ignore", category=RuntimeWarning)

if os.path.exists('bnc_examples.pkl'):
    with open('bnc_examples.pkl', 'rb') as f:
        bnc_examples = pickle.load(f)
else:
    bnc_examples = collect_bnc_examples(simlex999_wordlist)
    with open('bnc_examples.pkl', 'wb') as f:
        pickle.dump(bnc_examples, f)

if os.path.exists('all_layer_outputs.pkl'):
    with open('all_layer_outputs.pkl', 'rb') as f:
        all_layer_outputs = pickle.load(f)
else:
    all_layer_outputs = {}

    for i, word in enumerate(simlex999_wordlist):
        if i % 50 == 0:
            print(f"Processing word {i}/{len(simlex999_wordlist)}: {word}")
        sentences = bnc_examples[word]
        word_vectors = []  # each entry: (num_layers, hidden_dim)

        for sentence in sentences:
            word_start = sentence.index(word)
            word_end = word_start + len(word)

            enc = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=1024, return_offsets_mapping=True)
            offsets = enc["offset_mapping"][0]

            token_indices = [
                i for i, (tok_s, tok_e) in enumerate(offsets.tolist())
                if tok_s < word_end and tok_e > word_start and tok_s < tok_e
            ]

            inputs = {k: v.to(model.device) for k, v in enc.items() if k != "offset_mapping"}
            with torch.no_grad():
                model_output = model(**inputs, output_hidden_states=True)
                # hidden_states: tuple of (num_layers+1) tensors, each (batch, seq_len, hidden_dim)
                layer_vecs = torch.stack([
                    layer_hidden[0, token_indices, :].mean(dim=0)
                    for layer_hidden in model_output.hidden_states
                ])  # shape: (num_layers+1, hidden_dim)
                word_vectors.append(layer_vecs)

        word_vectors = [v for v in word_vectors if not v.isnan().any()]
        if word_vectors:
            # shape: (num_layers+1, hidden_dim)
            all_layer_outputs[word] = torch.stack(word_vectors).mean(dim=0)

    with open('all_layer_outputs.pkl', 'wb') as f:
        pickle.dump(all_layer_outputs, f)

Processing word 0/1998: old
Processing word 50/1998: bad
Processing word 100/1998: bold
Processing word 150/1998: lady
Processing word 200/1998: window
Processing word 250/1998: motor
Processing word 300/1998: boat
Processing word 350/1998: belief
Processing word 400/1998: machine
Processing word 450/1998: guy
Processing word 500/1998: wealth
Processing word 550/1998: money
Processing word 600/1998: molecule
Processing word 650/1998: bone
Processing word 700/1998: box
Processing word 750/1998: people
Processing word 800/1998: receive
Processing word 850/1998: speak
Processing word 900/1998: make
Processing word 950/1998: forget
Processing word 1000/1998: intelligent
Processing word 1050/1998: simple
Processing word 1100/1998: strange
Processing word 1150/1998: anchor
Processing word 1200/1998: wrist
Processing word 1250/1998: cab
Processing word 1300/1998: lunch
Processing word 1350/1998: illusion
Processing word 1400/1998: sea
Processing word 1450/1998: rice
Processing word 1500/1998:

In [63]:
#model_similarities = get_model_similarities(outputs)
    #in this case, we are going to want "model_layer_similarities", using get_model_similarities(_)

[0.58203125,
 0.7421875,
 0.7578125,
 0.796875,
 0.78515625,
 0.57421875,
 0.734375,
 0.75390625,
 0.7734375,
 0.89453125,
 0.73046875,
 0.69921875,
 0.8515625,
 0.71875,
 0.5078125,
 0.70703125,
 0.85546875,
 0.66015625,
 0.85546875,
 0.73046875,
 0.546875,
 0.6484375,
 0.6875,
 0.6875,
 0.8515625,
 0.87890625,
 0.796875,
 0.71484375,
 0.640625,
 0.50390625,
 0.875,
 0.84765625,
 0.59765625,
 0.59765625,
 0.85546875,
 0.5625,
 0.72265625,
 0.73828125,
 0.80078125,
 0.8359375,
 0.76953125,
 0.412109375,
 0.72265625,
 0.60546875,
 0.84375,
 0.53125,
 0.4765625,
 0.76171875,
 0.5,
 0.6171875,
 0.53515625,
 0.56640625,
 0.78515625,
 0.6484375,
 0.5625,
 0.625,
 0.671875,
 0.6484375,
 0.53125,
 0.80859375,
 0.66015625,
 0.734375,
 0.86328125,
 0.78515625,
 0.74609375,
 0.69921875,
 0.7265625,
 0.45703125,
 0.5625,
 0.578125,
 0.474609375,
 0.8671875,
 0.70703125,
 0.447265625,
 0.6640625,
 0.57421875,
 0.61328125,
 0.4765625,
 0.423828125,
 0.6015625,
 0.734375,
 0.57421875,
 0.40234375,
 

In [68]:
#spearmanr_correlations = spearmanr(model_similarities, human_similarities)
#print(spearmanr_correlations)

SignificanceResult(statistic=np.float64(0.3716142572113855), pvalue=np.float64(4.525726894714846e-34))
SignificanceResult(statistic=np.float64(0.4852946750573357), pvalue=np.float64(3.764123856042514e-60))


In [75]:
#Now to get the similarity correlation for each layer and inspect.
# "all_layer_outputs[word] will be a tensor of shape (37, 1280) — 37 because GPT-2 Large has 36 transformer layers
#  plus the initial embedding layer (index 0). To get the vector for a specific layer:
#  all_layer_outputs[word][layer_idx]."
# 37 total layers--36 transformer layer, and the embedding (indexed at 0).

for idx in range(37):
    print(f"Layer {idx} Spearman's p correlation is:" 
         f"{spearmanr(get_model_similarities({word: all_layer_outputs[word][idx] for word in all_layer_outputs}),
                                   human_similarities)}")
        

Layer 0 spearman's p correlation is:SignificanceResult(statistic=np.float64(0.4852946750573357), pvalue=np.float64(3.764123856042514e-60))
Layer 1 spearman's p correlation is:SignificanceResult(statistic=np.float64(0.3703736222409608), pvalue=np.float64(7.730726166549196e-34))
Layer 2 spearman's p correlation is:SignificanceResult(statistic=np.float64(0.3899369207120674), pvalue=np.float64(1.2631641857513437e-37))
Layer 3 spearman's p correlation is:SignificanceResult(statistic=np.float64(0.4149889637311034), pvalue=np.float64(7.312436932735697e-43))
Layer 4 spearman's p correlation is:SignificanceResult(statistic=np.float64(0.4365284507711105), pvalue=np.float64(9.805649971237096e-48))
Layer 5 spearman's p correlation is:SignificanceResult(statistic=np.float64(0.4568725342595568), pvalue=np.float64(1.1356160436809599e-52))
Layer 6 spearman's p correlation is:SignificanceResult(statistic=np.float64(0.46821552152348006), pvalue=np.float64(1.427081860441297e-55))
Layer 7 spearman's p cor